# Rolling half-life：从 lambda 到可执行计算

本课只讲 rolling half-life：它如何从 spread 的变化估计出来、lambda 的正负代表什么，以及为什么当前正在远离均值时仍可能看到 rolling half-life。

## 1. 共同前提

两条腿先按共同首日归一化，固定 beta 后定义：

$$S_t=Y_t-\beta X_t$$

生产环境里 $S_t$ 可以始终大于零。均值回复的参考点不是零，而是局部平衡位置 $\mu$；定义偏离为 $D_t=S_t-\mu$。

## 2. 从 lambda 到 half-life

先看最简单的局部关系：

$$\Delta S_t\approx\lambda D_{t-1}$$

- $D>0,\Delta S<0$ 或 $D<0,\Delta S>0$：正在拉回，$\lambda<0$。
- $D>0,\Delta S>0$ 或 $D<0,\Delta S<0$：继续远离，$\lambda>0$。

离散日线模型中，$D_t=(1+\lambda)D_{t-1}$。因此仅当 $-1<\lambda<0$ 时才有 half-life：

$$h_{1/2}=\frac{\ln(1/2)}{\ln(1+\lambda)}$$

它表示偏离从 $D$ 变为 $D/2$ 所需的交易日数。常见近似 $-\ln 2/\lambda$ 只在 $|\lambda|$ 很小时成立。若 $\lambda\ge0$ 或 $\lambda\le-1$，应记为 `NaN`／空白。

## 3. rolling lambda 实际拟合什么？

每个时点只取截至当天的 lookback 窗口，拟合带截距的 OLS：

$$\Delta S_t=a+\lambda S_{t-1}+\epsilon_t$$

截距让平衡位置可移动：$\mu=-a/\lambda$。所以等价地写成 $\Delta S_t=\lambda(S_{t-1}-\mu)+\epsilon_t$。


OLS 的 $\hat\lambda$ 不是 one-step lambda 的加总，而是窗口内水平与变化的回归斜率：

$$\hat\lambda=\frac{\sum (S_{t-1}-\overline S)(\Delta S_t-\overline{\Delta S})}{\sum(S_{t-1}-\overline S)^2}$$

## 4. Python：逐窗口计算 rolling lambda、mu 与 half-life

下面的 `lookback` 是每次回归使用的 price observations 数；回归中有 `lookback - 1` 个 $\Delta S$ 样本。每一个结果仅使用窗口结束日及之前的数据。

In [8]:
import numpy as np
import pandas as pd

def rolling_half_life(spread: pd.Series, lookback: int) -> pd.DataFrame:
    """每行只使用该行及之前的 lookback 个 spread observations。"""
    spread = pd.Series(spread, dtype=float).dropna()
    rows = []
    for end in range(lookback - 1, len(spread)):
        window = spread.iloc[end - lookback + 1 : end + 1].to_numpy()
        lagged = window[:-1]                 # S[t-1]
        delta = np.diff(window)             # Delta S[t]
        design = np.column_stack([np.ones(len(lagged)), lagged])
        intercept, lambda_ = np.linalg.lstsq(design, delta, rcond=None)[0]
        mu = -intercept / lambda_ if lambda_ != 0 else np.nan
        half_life = (np.log(0.5) / np.log1p(lambda_)
                     if -1 < lambda_ < 0 else np.nan)
        rows.append((spread.index[end], intercept, lambda_, mu, half_life))
    return pd.DataFrame(rows, columns=['date', 'a', 'lambda', 'mu', 'half_life']).set_index('date')


In [9]:
# 一个生产形状的例子：spread 始终为正，但围绕正的 mu 波动
theta = np.linspace(0, 2 * np.pi, 81)
spread = pd.Series(0.20 + 0.03 * np.sin(theta), index=pd.RangeIndex(len(theta)))
result = rolling_half_life(spread, lookback=16)
result[['lambda', 'mu', 'half_life']].tail(10)


,lambda,mu,half_life
date,,,
71,0.224581,0.169794,NaN
72,0.190323,0.169042,NaN
73,0.159919,0.168034,NaN
74,0.135002,0.166739,NaN
75,0.114711,0.165105,NaN
76,0.097966,0.163059,NaN
77,0.083888,0.160497,NaN
78,0.071821,0.157266,NaN
79,0.061286,0.153143,NaN


## 4.1 把 lambda hat 拆成五个零件

在一个窗口内，先不要看公式。把每一天的两列数据写出来：

- `x = S[t-1]`：昨天的 spread 水平；
- `y = Delta S[t]`：今天的变化。

然后依次做五件事：

1. 先算两列的窗口平均：

$$\bar{x}=\frac{1}{n}\sum_{i=1}^{n}x_i,\qquad \bar{y}=\frac{1}{n}\sum_{i=1}^{n}y_i$$

2. 对昨日水平去均值：`x_centered = x - x_mean`。

$$x_i^{c}=x_i-\bar{x}$$

它表示昨日水平比本窗口平均水平高还是低。

3. 对今日变化去均值：`y_centered = y - y_mean`。

$$y_i^{c}=y_i-\bar{y}$$

它表示今日变化比本窗口平均变化快还是慢。

4. 每行相乘：`cross = x_centered * y_centered`。

$$cross_i=x_i^{c}y_i^{c}$$

同号为正，表示高水平配较快上涨／低水平配较快下跌；异号为负，表示高水平配下跌／低水平配上涨。

5. 把所有交叉项相加，再除以昨日水平中心化平方和：

$$\hat{\lambda}=\frac{\sum_{i=1}^{n}cross_i}{\sum_{i=1}^{n}(x_i^{c})^2}=\frac{\sum_{i=1}^{n}(x_i-\bar{x})(y_i-\bar{y})}{\sum_{i=1}^{n}(x_i-\bar{x})^2}$$

分母永远非负，只负责把斜率按 spread 的波动尺度标准化。

### 本质：这就是协方差公式

更完整的专有名称是：**带截距简单线性回归的最小二乘斜率估计量**（OLS slope estimator），也常称为**回归系数**或**回归直线斜率**。在本课中，它就是 $\hat{\lambda}$。

令 $x=S_{t-1}$、$y=\Delta S_t$，上述带截距 OLS 回归的斜率恰好是：

$$\hat{\lambda}=\frac{\operatorname{Cov}(x,y)}{\operatorname{Var}(x)}=\frac{\operatorname{Cov}(S_{t-1},\Delta S_t)}{\operatorname{Var}(S_{t-1})}$$

这是一个完整的『协方差除以方差』结构，而不只是协方差公式：

- **分子** $\operatorname{Cov}(x,y)$：回答 $x$ 偏高／偏低时，$y$ 是否系统性同向或反向变化。它给出关系的方向与共同变化强度。
- **分母** $\operatorname{Var}(x)$：回答解释变量 $x$ 自己在窗口中有多大变化，并把分子按这个尺度标准化。
- **比值** $\hat\lambda$：每单位 $x$ 的偏离，对应的平均 $y$ 改变，即回归直线的斜率。

协方差与方差若都使用样本版（分母 $n-1$）或都使用总体版（分母 $n$），两者的归一化因子会相消，因此斜率相同。

这个等价关系依赖回归中包含截距 $a$，因为截距对应先对两列去均值。若强行拟合无截距模型 $\Delta S_t=\lambda S_{t-1}$，斜率会变成 $\sum xy / \sum x^2$，不再是这里的中心化协方差公式。

In [10]:
def lambda_hat_parts(spread_window: pd.Series):
    """展示一个窗口内 OLS lambda 的每一个零件。"""
    s = pd.Series(spread_window, dtype=float).reset_index(drop=True)
    x = s.iloc[:-1].to_numpy()
    y = np.diff(s.to_numpy())
    x_centered = x - x.mean()
    y_centered = y - y.mean()
    parts = pd.DataFrame({
        'S[t-1]': x, 'Delta S[t]': y,
        'x_centered': x_centered, 'y_centered': y_centered,
        'cross': x_centered * y_centered,
        'x_centered_sq': x_centered**2,
    })
    lambda_hat = parts['cross'].sum() / parts['x_centered_sq'].sum()
    return parts, lambda_hat

parts, lambda_hat = lambda_hat_parts(spread.iloc[:16])
display(parts)
print('lambda_hat =', lambda_hat)


,S[t-1],Delta S[t],x_centered,y_centered,cross,x_centered_sq
0,0.200000,0.002354,-0.014788,0.000506,-7.482870e-06,2.186813e-04
1,0.202354,0.002339,-0.012434,0.000492,-6.111387e-06,1.546069e-04
2,0.204693,0.002310,-0.010095,0.000463,-4.669550e-06,1.019058e-04
3,0.207003,0.002267,-0.007785,0.000419,-3.264747e-06,6.059868e-05
4,0.209271,0.002210,-0.005517,0.000362,-1.998578e-06,3.044133e-05
5,0.211481,0.002139,-0.003307,0.000291,-9.639437e-07,1.093872e-05
6,0.213620,0.002055,-0.001168,0.000207,-2.423735e-07,1.364601e-06
7,0.215675,0.001959,0.000887,0.000111,9.832541e-08,7.869121e-07
8,0.217634,0.001850,0.002846,0.000002,6.046549e-09,8.097902e-06
9,0.219483,0.001730,0.004696,-0.000118,-5.540630e-07,2.204833e-05


lambda_hat = -0.049642556616730595


## 4.2 重要限制：OLS half-life 不是正弦的真实衰减时间

最小二乘确实是在拟合散点，但它做的是一个**最佳线性投影**：在所有形如 $a+\lambda S_{t-1}$ 的直线中，找平方误差最小的一条。它并不证明真实过程本身就是一维线性均值回复。

纯正弦 $D_t=A\sin(\omega t)$ 永远不会衰减；它的真实离散动力学需要前两期状态：

$$D_{t+1}=2\cos(\omega)D_t-D_{t-1}$$

这是二阶振荡系统，不是 $D_t=(1+\lambda)D_{t-1}$ 这种一阶指数衰减系统。给正弦拟合 $\Delta D_t=a+\lambda D_{t-1}+\epsilon_t$ 时，遗漏的相位／速度信息会进入误差项。

即使在完整、均匀的一个周期上，OLS 仍可能给出负斜率：

$$\hat\lambda=\cos(\omega)-1<0$$

这只是『用一条一阶直线最好地概括一个周期的散点』的结果，不表示正弦振幅会减半。因此对正弦显示的 half-life 应叫作**AR(1) 线性代理时间尺度**，用于理解拟合器的行为；不能当作正弦的周期或物理衰减常数。

在真实 spread 上，half-life 的可解释性取决于一阶 AR／OU 近似在该窗口是否足够好。应结合残差、自相关稳定性和样本外表现验证，而不要仅凭 $\hat\lambda<0$ 就断言存在真实均值回复。

## 4.3 从 lambda 到 life：log 2 从哪里来？

这一步需要额外作出一个**模型假设**：把刚估计出的 $\hat\lambda$ 当作接下来每一步都不变的一阶 AR 系数。令偏离均衡位置的量为 $D_t=S_t-\mu$，则预测模型为：

$$D_{t+1}=(1+\hat\lambda)D_t$$

连续迭代 $k$ 次：

$$D_{t+k}=(1+\hat\lambda)^kD_t$$

现在不必先问“减半”。先选一个目标比例 $r$，例如 $r=1/2$，并问偏离何时变为原来的 $r$ 倍：

$$\frac{D_{t+k}}{D_t}=r=(1+\hat\lambda)^k$$

对两边取对数并解 $k$，得到通用的时间尺度：

$$\operatorname{life}_{r}=\frac{\ln r}{\ln(1+\hat\lambda)},\qquad -1<\hat\lambda<0$$

上式的单位是**样本步数**。若相邻样本间隔为 $\Delta t$，真正的时间单位为：

$$T_r=\Delta t\cdot\operatorname{life}_{r}$$

所以日线输入得到交易日数；5 分钟 bar 输入得到 5 分钟 bar 的个数，再乘 5 分钟才是钟表时间。时间维度不是 OLS 凭空创造的，而是来自每一个 $t\to t+1$ 样本步的真实间隔。

因此：

$$\operatorname{life}_{1/2}=\frac{\ln(1/2)}{\ln(1+\hat\lambda)},\qquad \operatorname{life}_{1/3}=\frac{\ln(1/3)}{\ln(1+\hat\lambda)},\qquad \operatorname{life}_{1/10}=\frac{\ln(1/10)}{\ln(1+\hat\lambda)}$$

这就是 log 2 的来源：life_1/2 的目标比例恰好是 $1/2$，而不是某个任意的对数常数。改成 $1/3$、$1/10$ 是不同的衰减目标；它们有意义，但名称分别是 third-life、tenth-life，而不是 half-life。对数底数本身不重要：分子、分母使用同一种底数时会相消。

若 $|\hat\lambda|$ 很小，$\ln(1+\hat\lambda)\approx\hat\lambda$，才得到常见近似 $\operatorname{life}_{1/2}\approx-\ln 2/\hat\lambda$。

In [11]:
def life_to_fraction(lambda_: float, fraction: float) -> float:
    """AR(1) 代理模型中，偏离缩小到 fraction 所需的样本步数。"""
    if not (-1 < lambda_ < 0 and 0 < fraction < 1):
        return np.nan
    return np.log(fraction) / np.log1p(lambda_)

lambda_example = -0.10
pd.Series({
    'life_1/2': life_to_fraction(lambda_example, 1/2),
    'life_1/3': life_to_fraction(lambda_example, 1/3),
    'life_1/10': life_to_fraction(lambda_example, 1/10),
})


life_1/2      6.578813
life_1/3     10.427173
life_1/10    21.854345
dtype: float64

### 正弦为什么也会产生一个 life？

对每一步相位增量为 $\omega$ 的均匀正弦样本，完整周期的 OLS 投影给出 $\hat\lambda=\cos(\omega)-1$。把它代入上式，会得到一个数值上的 $\operatorname{life}_{1/2}$。这是因为我们先把正弦散点投影为一阶 AR(1)，再计算那个 AR(1) 代理模型的时间尺度。

它不是正弦自己的 half-life：正弦不衰减。下例会显示，100 个点走完一整个周期的正弦，AR(1) 代理却可能给出约 351 个样本点的 life_1/2；这种明显不一致正好说明它只是模型投影，不是物理事实。

In [12]:
omega = 2 * np.pi / 100  # 101 points: index 0..100 over one full sine cycle
lambda_sine_proxy = np.cos(omega) - 1
pd.Series({
    'sine_OLS_lambda_hat': lambda_sine_proxy,
    'AR_proxy_life_1/2': life_to_fraction(lambda_sine_proxy, 1/2),
    'sine_period_in_samples': 100,
})


sine_OLS_lambda_hat        -0.001973
AR_proxy_life_1/2         350.921324
sine_period_in_samples    100.000000
dtype: float64

## 4.4 逐渐变慢的正弦：只看瞬时 lambda 的收缩条件

令正弦的相位为 $\phi(t)$，局部频率为 $\omega(t)=\phi'(t)$：

$$D_t=A\sin\phi(t),\qquad S_t=\mu+D_t$$

若要让频率逐渐变慢，必须先积分频率：

$$\phi(t)=\omega_0t+\frac{\omega_1-\omega_0}{2T}t^2,\qquad \omega(t)=\omega_0+\frac{\omega_1-\omega_0}{T}t$$

这里 $\omega_1<\omega_0$。不应写成 $\sin(\omega(t)t)$，因为它的实际相位速度会额外包含 $t\omega'(t)$。\n

暂时不要做 rolling OLS，也不要计算 half-life。每一步只计算：

$$\lambda_t^{\mathrm{inst}}=\frac{\Delta D_t}{D_{t-1}}=\frac{\sin\phi(t)-\sin\phi(t-1)}{\sin\phi(t-1)}$$

若一个很短区间内的相位步长可视为常数 $\delta\phi$，可进一步拆开为：

$$\lambda^{\mathrm{inst}}=(\cos\delta\phi-1)+\sin(\delta\phi)\cot\phi_{t-1}$$

这说明瞬时 lambda 同时受**局部频率**（$\delta\phi$）与**当前相位**（$\phi_{t-1}$）影响。因此即使频率持续变慢，沿时间画出的 lambda 仍会随相位上下起伏；要比较频率效应，应在不同周期的相同相位位置比较。\n

这个比值只有在下列条件同时成立时，才可称为一步收缩：

$$-1<\lambda_t^{\mathrm{inst}}<0$$

其中负号表示朝平衡位置移动；下界 $-1$ 保证一步之后仍在同侧、偏离幅度缩小，而不是越过平衡位置。等价地：

$$D_{t-1}\Delta D_t<0,\qquad |D_t|<|D_{t-1}|$$

当每一步相位增量很小，并在同一个相位位置比较时：

$$\lambda_t^{\mathrm{inst}}\approx\omega(t)\cot\phi(t)$$

在回归波段中 $\cot\phi(t)<0$。因此在**固定物理采样间隔**下，频率变慢、$\omega(t)$ 变小时，瞬时 $\lambda$ 的绝对值应趋于变小、向 0 靠近。\n

但若为了受控比较而规定**每个周期恒有相同的 N 个相位采样点**，则 $\delta\phi=2\pi/N$ 被固定。此时 $\lambda^{\mathrm{inst}}$ 是“每相位样本步”的量，主要反映相位位置，不再随物理频率变化。下面用同一条 $\phi(t)=t^3$ 的曲线并排展示等时间采样与等相位采样；前者模拟 K 线，后者只是对照。

In [ ]:
# 同一条 phi(t)=t^3 曲线：等时间 K 线式采样 vs 每周期等相位采样。\n
cycles, points_per_cycle, time_intervals = 8, 100, 800\n
t_end = (2 * np.pi * cycles)**(1/3)\n
t_time = np.linspace(0, t_end, time_intervals + 1)\n
phase_time = t_time**3\n
phase_equal = (2 * np.pi / points_per_cycle) * np.arange(cycles * points_per_cycle + 1)\n
t_equal = phase_equal**(1/3)\n
\n
def contracting_lambda(values, mean=0.20):\n
    deviation = values - mean\n
    out = np.full(len(values), np.nan)\n
    np.divide(np.diff(values), deviation[:-1], out=out[1:],\n
              where=np.abs(deviation[:-1]) > 1e-12)\n
    out[(out <= -1) | (out >= 0)] = np.nan\n
    return out\n
\n
spread_time = 0.20 + 0.03 * np.sin(phase_time)\n
spread_equal = 0.20 + 0.03 * np.sin(phase_equal)\n
pd.DataFrame({'t_equal_time': t_time,\n
              'lambda_equal_time': contracting_lambda(spread_time)})\n
pd.DataFrame({'t_equal_phase': t_equal,\n
              'lambda_equal_phase': contracting_lambda(spread_equal)})\n

## 5. 为什么 one-step 为正，rolling lambda 仍可能为负？

one-step 的方向参考应为 $\Delta S_t/(S_{t-1}-\mu)$，而不是单看 $\Delta S_t$。在纯中心化正弦的特殊例子中，$\mu=0$，两种写法看起来相同；生产 spread 则通常必须显式减去正的 $\mu$。

即使若干 one-step 方向为正，rolling OLS 仍可能为负：它拟合的是整个窗口的协方差，并且同时估计局部截距与局部 $\mu$。窗口越长，越可能混入此前的拉回段；窗口越短，越贴近当前方向但越嘈杂。

若 **rolling 回归本身** 估计到 $\hat\lambda\ge0$，代码会返回 `NaN`，half-life 图必须留空。

## 6. 一个周期的验证实验

黑板 `half_life_and_across_density_blackboard.nb` 现在使用一个从 $0$ 到 $2\pi$ 的正弦周期：

1. 第二图：黄色 one-step 方向参考与粉红 rolling lambda。
2. 第三图：$S-\mu$，明确当前点是在均值上方还是下方。
3. 第四图：只在 rolling lambda 为负时出现 half-life。
4. 把窗口从 16 调短，观察 rolling lambda 如何更快跟随当前相位。

## 练习

1. 对一个永远大于零、但围绕 $\mu=0.2$ 波动的 spread，为什么仍可有 $\lambda<0$？
2. 当 one-step direction 为正而 rolling lambda 为负时，分别检查窗口长度、局部 $\mu$ 和协方差。
3. 把 `lookback` 从 60 改为 15，half-life 曲线会有什么权衡？